<a href="https://colab.research.google.com/github/ritabrezznica/DeepLearning/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:

df = pd.read_excel("/content/drive/MyDrive/default of credit card clients.xls", header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [5]:
df = df.drop(columns=['ID'])
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)
print(df[['EDUCATION', 'MARRIAGE']].head())

   EDUCATION  MARRIAGE
0          2         1
1          2         2
2          2         2
3          2         1
4          2         1


In [6]:
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [7]:
X = df.drop(columns=['default payment next month'])
y = df['default payment next month']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f" Raw Train Shape: {X_train.shape}")

 Raw Train Shape: (24000, 23)


In [10]:
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']

In [13]:
X_train_cat = pd.get_dummies(X_train[cat_cols], columns=cat_cols)
X_test_cat = pd.get_dummies(X_test[cat_cols], columns=cat_cols)
print(f"Post One-Hot Shape: {X_train_cat.shape}")

Post One-Hot Shape: (24000, 9)


In [14]:
X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join='left', axis=1, fill_value=0)

In [15]:
X_train_bipolar = X_train_cat.astype(float).replace({0: -1.0})
X_test_bipolar = X_test_cat.astype(float).replace({0: -1.0})

In [17]:
print(f"Bipolar Shape: {X_train_bipolar.shape}")
print(f"Sample Bipolar Row:\n{X_train_bipolar.iloc[0].values[:5]}...")

Bipolar Shape: (24000, 9)
Sample Bipolar Row:
[-1.  1. -1.  1. -1.]...


In [21]:
num_cols = [col for col in X.columns if col not in cat_cols]
print(f"Scaling {len(num_cols)} numerical columns...")


Scaling 20 numerical columns...


In [22]:
scaler = StandardScaler()  # <--- This is the missing piece!
X_train_num_scaled = scaler.fit_transform(X_train[num_cols])
X_test_num_scaled = scaler.transform(X_test[num_cols])

print(f"Step 4: Scaled Mean (should be ~0): {X_train_num_scaled.mean():.4f}")

Step 4: Scaled Mean (should be ~0): -0.0000


In [23]:
X_train_final = np.hstack([X_train_num_scaled, X_train_bipolar.values])
X_test_final = np.hstack([X_test_num_scaled, X_test_bipolar.values])

print("Final Preprocessing Complete ")
print(f"Final Train Shape: {X_train_final.shape}")
print(f"Final Test Shape: {X_test_final.shape}")

print(f"\nStructure Check:")
print(f"Numerical part: {X_train_num_scaled.shape[1]} columns")
print(f"Bipolar part:   {X_train_bipolar.shape[1]} columns")
print(f"Total:          {X_train_final.shape[1]} columns")

print("\nFirst row sample:")
print(X_train_final[0])

Final Preprocessing Complete 
Final Train Shape: (24000, 29)
Final Test Shape: (6000, 29)

Structure Check:
Numerical part: 20 columns
Bipolar part:   9 columns
Total:          29 columns

First row sample:
[-0.05686623 -0.26455769  1.79331103  1.78019315  2.65204636  1.91181115
  0.2402604   0.25608731  1.50554693  1.74508934  1.77886926  1.89167911
  2.02083925  2.09634558  0.58065737 -0.29033241 -0.29781997  0.08696116
  0.50039738  0.04874486 -1.          1.         -1.          1.
 -1.         -1.         -1.          1.         -1.        ]
